In [ ]:


!pip -q install -U sentence-transformers faiss-cpu transformers accelerate pypdf



import os
import re
import json
import pickle
import time
from pathlib import Path

import numpy as np
import torch
import faiss

from pypdf import PdfReader
from google.colab import files
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    GenerationConfig,
    pipeline,
)



USAR_DRIVE = True
FORCAR_REINDEXAR = False

MODELO_EMBEDDING = (
    "sentence-transformers/"
    "paraphrase-multilingual-MiniLM-L12-v2"
)

MODELO_GERADOR = "Qwen/Qwen2.5-1.5B-Instruct"

# Parâmetros de desempenho
TAMANHO_CHUNK = 1200
SOBREPOSICAO_CHUNK = 200
BATCH_SIZE_EMBEDDINGS = 32

# Menos trechos significa menos contexto para o gerador.
K_DOCUMENTOS = 3

# Limite de tokens novos por resposta.
MAX_TOKENS_RESPOSTA = 160

# Verificar hardware disponível
USAR_GPU = torch.cuda.is_available()

if USAR_GPU:
    print("GPU detectada:", torch.cuda.get_device_name(0))
    TIPO_DADOS = torch.float16
else:
    print("GPU CUDA indisponível. Usando CPU.")
    TIPO_DADOS = torch.float32

print("Versão do PyTorch:", torch.__version__)
print("CUDA disponível:", USAR_GPU)



if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DIR_PERSISTENCIA = Path("/content/drive/MyDrive/rag_store")
else:
    DIR_PERSISTENCIA = Path("/content/rag_store")

DIR_PERSISTENCIA.mkdir(parents=True, exist_ok=True)

ARQ_EMBEDDINGS = DIR_PERSISTENCIA / "embeddings.npy"
ARQ_INDICE = DIR_PERSISTENCIA / "indice.faiss"
ARQ_CHUNKS = DIR_PERSISTENCIA / "chunks.pkl"
ARQ_METADADOS = DIR_PERSISTENCIA / "metadados.json"
ARQ_CONFIG = DIR_PERSISTENCIA / "config.json"



def extrair_paginas_pdf(caminho):
    """Extrai o texto de cada página do PDF."""

    reader = PdfReader(caminho)
    paginas = []
    paginas_sem_texto = []

    for numero, pagina in enumerate(reader.pages, start=1):
        try:
            texto = pagina.extract_text() or ""
        except Exception as erro:
            print(
                f"Erro ao extrair {caminho}, "
                f"página {numero}: {erro}"
            )
            texto = ""

        texto = re.sub(r"-\s*\n\s*(\w)", r"\1", texto)
        texto = re.sub(r"[ \t]+", " ", texto)
        texto = re.sub(r"\n{3,}", "\n\n", texto).strip()

        if len(texto) < 30:
            paginas_sem_texto.append(numero)

        if texto:
            paginas.append({
                "arquivo": caminho,
                "pagina": numero,
                "texto": texto,
            })

    if paginas_sem_texto:
        print(
            f"Aviso: {caminho} contém páginas "
            f"sem texto extraível ou quase vazias: "
            f"{paginas_sem_texto[:20]}"
        )

        print(
            "Se houver páginas digitalizadas como imagens, "
            "pode ser necessário aplicar OCR."
        )

    return paginas




def criar_chunks(
    texto,
    tamanho=TAMANHO_CHUNK,
    sobreposicao=SOBREPOSICAO_CHUNK,
):
    """Divide o texto em trechos de tamanho aproximado."""

    paragrafos = [
        p.strip()
        for p in re.split(r"\n\s*\n", texto)
        if p.strip()
    ]

    chunks = []
    atual = ""

    for paragrafo in paragrafos:

        if len(paragrafo) > tamanho:
            if atual:
                chunks.append(atual)
                atual = ""

            inicio = 0

            while inicio < len(paragrafo):
                trecho = paragrafo[
                    inicio:inicio + tamanho
                ].strip()

                if trecho:
                    chunks.append(trecho)

                inicio += tamanho - sobreposicao

            continue

        candidato = (
            f"{atual}\n\n{paragrafo}".strip()
            if atual
            else paragrafo
        )

        if len(candidato) <= tamanho:
            atual = candidato
        else:
            if atual:
                chunks.append(atual)

            atual = paragrafo

    if atual:
        chunks.append(atual)

    return chunks



def salvar_estado(
    documentos,
    metadados,
    embeddings,
    indice,
    modelo_embedding_nome,
    caminhos_pdf,
):
    """Salva os dados necessários para reutilizar o índice."""

    np.save(ARQ_EMBEDDINGS, embeddings)

    faiss.write_index(indice, str(ARQ_INDICE))

    with open(ARQ_CHUNKS, "wb") as f:
        pickle.dump(
            documentos,
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )

    with open(ARQ_METADADOS, "w", encoding="utf-8") as f:
        json.dump(
            metadados,
            f,
            ensure_ascii=False,
            indent=2,
        )

    config = {
        "modelo_embedding": modelo_embedding_nome,
        "dimensao": int(embeddings.shape[1]),
        "num_vetores": int(embeddings.shape[0]),
        "num_documentos": len(documentos),
        "caminhos_pdf": list(caminhos_pdf),
    }

    with open(ARQ_CONFIG, "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print("\n[PERSISTÊNCIA] Arquivos salvos:")

    for arq in [
        ARQ_EMBEDDINGS,
        ARQ_INDICE,
        ARQ_CHUNKS,
        ARQ_METADADOS,
        ARQ_CONFIG,
    ]:
        tamanho_mb = arq.stat().st_size / (1024 * 1024)

        print(f"  {arq.name}: {tamanho_mb:.2f} MB")




def carregar_estado(forcar_reindexar=False):

    if forcar_reindexar:
        return None

    necessarios = [
        ARQ_EMBEDDINGS,
        ARQ_INDICE,
        ARQ_CHUNKS,
        ARQ_METADADOS,
        ARQ_CONFIG,
    ]

    if not all(arq.exists() for arq in necessarios):
        print(
            "[PERSISTÊNCIA] Índice anterior não encontrado. "
            "Será necessário indexar os PDFs."
        )
        return None

    with open(ARQ_CONFIG, "r", encoding="utf-8") as f:
        config = json.load(f)

    print("\n[PERSISTÊNCIA] Estado encontrado:")
    print("Modelo:", config["modelo_embedding"])
    print("Vetores:", config["num_vetores"])
    print("PDFs:", config["caminhos_pdf"])

    indice = faiss.read_index(str(ARQ_INDICE))

    with open(ARQ_CHUNKS, "rb") as f:
        documentos = pickle.load(f)

    with open(ARQ_METADADOS, "r", encoding="utf-8") as f:
        metadados = json.load(f)

    modelo_embedding = SentenceTransformer(
        config["modelo_embedding"]
    )

    assert (
        indice.ntotal == len(documentos) == len(metadados)
    ), (
        f"Inconsistência: índice={indice.ntotal}, "
        f"chunks={len(documentos)}, "
        f"metadados={len(metadados)}"
    )

    print("[PERSISTÊNCIA] Índice carregado com sucesso.")

    return (
        documentos,
        metadados,
        indice,
        modelo_embedding,
    )




inicio_inicializacao = time.perf_counter()

estado = carregar_estado(
    forcar_reindexar=FORCAR_REINDEXAR
)

if estado is None:

    print("\nSelecione um ou mais arquivos PDF:")

    uploaded = files.upload()
    caminhos_pdf = list(uploaded.keys())

    if not caminhos_pdf:
        raise ValueError("Nenhum PDF foi enviado.")

    paginas_pdf = []

    for caminho in caminhos_pdf:
        paginas = extrair_paginas_pdf(caminho)
        paginas_pdf.extend(paginas)

        print(
            f"{caminho}: {len(paginas)} "
            "páginas com texto extraído."
        )

    if not paginas_pdf:
        raise ValueError(
            "Nenhum texto foi extraído dos PDFs."
        )

    documentos = []
    metadados = []

    for pagina in paginas_pdf:
        chunks = criar_chunks(pagina["texto"])

        for chunk in chunks:
            documentos.append(chunk)

            metadados.append({
                "arquivo": pagina["arquivo"],
                "pagina": pagina["pagina"],
            })

    print("\nTotal de chunks:", len(documentos))

    if not documentos:
        raise ValueError(
            "Não foi possível criar trechos para indexação."
        )

    print("Carregando modelo de embeddings...")

    modelo_embedding = SentenceTransformer(
        MODELO_EMBEDDING
    )

    print("Calculando embeddings...")

    embeddings = modelo_embedding.encode(
        documentos,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
        batch_size=BATCH_SIZE_EMBEDDINGS,
    ).astype("float32")

    # Similaridade cosseno por produto interno,
    # pois os embeddings estão normalizados.
    indice = faiss.IndexFlatIP(
        embeddings.shape[1]
    )

    indice.add(embeddings)

    salvar_estado(
        documentos=documentos,
        metadados=metadados,
        embeddings=embeddings,
        indice=indice,
        modelo_embedding_nome=MODELO_EMBEDDING,
        caminhos_pdf=caminhos_pdf,
    )

else:
    (
        documentos,
        metadados,
        indice,
        modelo_embedding,
    ) = estado

print(
    f"\nÍndice pronto: {indice.ntotal} "
    "trechos indexados."
)

print(
    "Tempo de preparação do índice: "
    f"{time.perf_counter() - inicio_inicializacao:.2f} s"
)



def recuperar_documentos(pergunta, k=K_DOCUMENTOS):

    k_real = min(max(1, k), len(documentos))

    vetor_pergunta = modelo_embedding.encode(
        [pergunta],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    similaridades, indices = indice.search(
        vetor_pergunta,
        k_real,
    )

    resultados = []

    for idx, score in zip(
        indices[0],
        similaridades[0],
    ):
        if idx < 0:
            continue

        resultados.append({
            "documento": documentos[idx],
            "arquivo": metadados[idx]["arquivo"],
            "pagina": metadados[idx]["pagina"],
            "similaridade": float(score),
        })

    return resultados



print("\nCarregando modelo gerador...")

inicio_modelo = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(
    MODELO_GERADOR
)

modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_GERADOR,
    torch_dtype=TIPO_DADOS,
    device_map="auto",
    low_cpu_mem_usage=True,
)

modelo.eval()

generation_config = GenerationConfig.from_pretrained(
    MODELO_GERADOR,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    max_new_tokens=MAX_TOKENS_RESPOSTA,
    do_sample=False,
    repetition_penalty=1.05,
    use_cache=True,
)

gerador = pipeline(
    "text-generation",
    model=modelo,
    tokenizer=tokenizer,
    generation_config=generation_config,
)

print(
    "Modelo carregado em "
    f"{time.perf_counter() - inicio_modelo:.2f} s"
)

print("Dispositivo do modelo:", modelo.device)



def rag(
    pergunta,
    k=K_DOCUMENTOS,
    max_new_tokens=MAX_TOKENS_RESPOSTA,
    mostrar_contexto=False,
):
    """Recupera trechos e gera uma resposta com fontes."""

    inicio_total = time.perf_counter()



    resultados = recuperar_documentos(
        pergunta,
        k=k,
    )

    tempo_recuperacao = (
        time.perf_counter() - inicio_total
    )

    if not resultados:
        return (
            "Não encontrei trechos relevantes nos documentos.",
            [],
        )



    blocos_contexto = []

    for i, r in enumerate(resultados, start=1):
        blocos_contexto.append(
            f"[Trecho {i} | Arquivo: {r['arquivo']} | "
            f"Página: {r['pagina']}]\n"
            f"{r['documento']}"
        )

    contexto = "\n\n---\n\n".join(
        blocos_contexto
    )



    mensagens = [
        {
            "role": "system",
            "content": (
                "Você é um assistente de perguntas e respostas "
                "sobre documentos PDF. Responda em português claro "
                "e diretamente à pergunta. Use somente as informações "
                "do CONTEXTO fornecido. Não invente fatos, números, "
                "autores, datas ou referências. Se o contexto não "
                "contiver informação suficiente, diga explicitamente: "
                "'Não encontrei informação suficiente nos trechos "
                "recuperados para responder com segurança.' "
                "Diferencie o que o documento afirma de inferências. "
                "Ao final de cada afirmação factual importante, "
                "indique a fonte no formato [arquivo, p. X]. "
                "Se as fontes divergirem, informe a diferença."
            ),
        },
        {
            "role": "user",
            "content": (
                f"CONTEXTO:\n{contexto}\n\n"
                f"PERGUNTA:\n{pergunta}"
            ),
        },
    ]

    prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True,
    )


    inicio_geracao = time.perf_counter()


    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with torch.inference_mode():
        saida = gerador(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            return_full_text=False,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    resposta = saida[0]["generated_text"].strip()

    tempo_geracao = (
        time.perf_counter() - inicio_geracao
    )

    tempo_total = (
        time.perf_counter() - inicio_total
    )



    print("\n" + "=" * 70)
    print("PERGUNTA:", pergunta)
    print("=" * 70)

    print("\nRESPOSTA:\n")
    print(resposta)

    print("\nTRECHOS RECUPERADOS:")

    for r in resultados:
        print(
            f"- {r['arquivo']} | página {r['pagina']} | "
            f"similaridade cosseno={r['similaridade']:.3f}"
        )

    print("\nTEMPOS DE EXECUÇÃO")
    print(f"Recuperação: {tempo_recuperacao:.2f} s")
    print(f"Geração:     {tempo_geracao:.2f} s")
    print(f"Total:       {tempo_total:.2f} s")

    if mostrar_contexto:
        print("\nCONTEXTO ENVIADO AO MODELO:\n")
        print(contexto)

    return resposta, resultados



print("\nSistema RAG pronto!")
print("Digite 'sair' para encerrar.")

while True:

    pergunta = input(
        "\nFaça uma pergunta sobre os PDFs: "
    ).strip()

    if pergunta.lower() in {
        "sair",
        "exit",
        "quit",
    }:
        print("Encerrando o sistema.")
        break

    if not pergunta:
        print("Digite uma pergunta.")
        continue

    try:
        rag(
            pergunta,
            k=K_DOCUMENTOS,
            max_new_tokens=MAX_TOKENS_RESPOSTA,
        )

    except Exception as erro:
        print(f"Erro ao responder: {erro}")
        print(
            "Se ocorrer falta de memória, considere usar "
            "Qwen/Qwen2.5-0.5B-Instruct."
        )


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 kB 32.4 MB/s eta 0:00:00
GPU detectada: Tesla T4
Versão do PyTorch: 2.11.0+cu130
CUDA disponível: True
Mounted at /content/drive

[PERSISTÊNCIA] Estado encontrado:
Modelo: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Vetores: 104
PDFs: ['15+61959+FINAL+2026+10+09 (2) (1).pdf']


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Modelo carregado em 35.91 s
Dispositivo do modelo: cuda:0

Sistema RAG pronto!
Digite 'sair' para encerrar.

Faça uma pergunta sobre os PDFs: O que é RAG?


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'use_cache', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



PERGUNTA: O que é RAG?

RESPOSTA:

RAG é uma técnica utilizada para melhorar a confiabilidade de modelos de IA generativa, utilizando dados obtidos de fontes externas. Esta técnica busca mitigar alguns dos problemas dos LLMs tradicionais, como a existência de dados desatualizados, falta de precisão em domínios específicos, alucinações, entre outros aspectos. [arquivo, p. 3]

TRECHOS RECUPERADOS:
- 15+61959+FINAL+2026+10+09 (2) (1).pdf | página 3 | similaridade cosseno=0.437
- 15+61959+FINAL+2026+10+09 (2) (1).pdf | página 4 | similaridade cosseno=0.427
- 15+61959+FINAL+2026+10+09 (2) (1).pdf | página 26 | similaridade cosseno=0.376

TEMPOS DE EXECUÇÃO
Recuperação: 1.15 s
Geração:     3.87 s
Total:       5.04 s

Faça uma pergunta sobre os PDFs: O que é CVD?

PERGUNTA: O que é CVD?

RESPOSTA:

O CVD, ou Ciclo de Vida de Dados, é uma ferramenta abrangente e flexível que trata das fases de acesso aos dados, desde a coleta, uso até o descarte, considerando aspectos importantes como a quali